<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# 🎧 AfriCare Support Analytics — DataProjectLab

## 📓 Notebook 4 — Dashboard Power BI & Storytelling

---

> **🎯 Niveau :** Avancé | **⏱️ Durée :** 3-4h
> ❗ Ce notebook ne contient pas de solution.

### Objectif
Transformer les analyses du Notebook 3 en un **outil de pilotage visuel** opérationnel, lisible et actionnable.  
Le dashboard doit permettre à M. KOUAME et à ses superviseurs de prendre des décisions en quelques minutes.

---

# 1. Objectif du dashboard

Le dashboard doit aider la direction et les superviseurs à piloter :
- le volume de tickets et les tendances
- le respect des SLA par catégorie et par agent
- la détection des tickets à risque (issus du Notebook 5 ML)
- la satisfaction client (CSAT)
- la performance individuelle des agents

Le bon niveau de lecture est :  
👉 **simple à comprendre, rapide à lire, actionnable**.

---

# 2. Architecture recommandée — 6 pages

## Page 1 — Alertes & risques (temps réel)

### Visuels à construire
- Bandeau rouge dynamique : "X tickets nécessitent une intervention immédiate"
- Tableau `tickets_risque_scores.csv` filtré sur ROUGE + ORANGE
- KPI Cards : Tickets ouverts · Score risque moyen · Nb ROUGE · Nb ORANGE
- Donut : répartition des niveaux d'alerte (ROUGE / ORANGE / JAUNE / VERT)
- Barres : score de risque moyen par catégorie

### Message business
> Cette page doit permettre aux superviseurs de décider **immédiatement** quels tickets nécessitent une intervention.

### ✏️ Ton travail
Connecte `tickets_risque_scores.csv` dans Power BI. Construis le tableau avec formatage conditionnel (fond rouge/orange/jaune/vert selon `niveau_alerte`).

---

## Page 2 — Performance SLA & délais

### Visuels à construire
- KPI Cards : Taux SLA breach · First response moy · Résolution moy · Catégorie critique
- Courbe mensuelle : taux breach avec variation MoM
- Barres par catégorie : taux breach (couleur conditionnelle rouge/orange/vert)
- Heatmap Matrice : heure × jour · volume de tickets
- Barres : distribution des dépassements SLA par tranche

### Message business
> Identifier immédiatement les catégories et les périodes qui posent problème.

---

## Page 3 — Performance des agents

### Visuels à construire
- KPI Cards : CSAT global · Top performer · Coaching urgent · Charge max
- Scatter Breach vs CSAT (taille = charge, couleur = tier) avec quadrants
- Tableau agents : Tier · Breach % · CSAT · Escalades · Statut (badge coloré)
- Barres : charge (nb tickets traités) par agent
- Barres groupées : taux d'escalade par tier

### Message business
> Distinguer les agents performants des agents qui nécessitent un accompagnement.

---

## Page 4 — Volume & backlog

### Visuels à construire
- KPI Cards : Total tickets · Taux backlog · Canal principal · Impact weekend
- Barres empilées mensuelles : Résolu / En cours / Backlog / Escaladé
- Barres + courbe : volume par canal avec % SLA breach (axe secondaire)
- Barres : volume par pays
- Barres horizontales : taux backlog par catégorie

### Message business
> Surveiller la formation du backlog et les effets de saisonnalité.

---

## Page 5 — Satisfaction client

### Visuels à construire
- KPI Cards : CSAT moyen · % avis négatifs · Impact délai long · % tickets avec CSAT
- Barres : distribution des notes 1-5 (coloré rouge→vert)
- Courbe : CSAT moyen vs délai de résolution (tranches)
- Barres : CSAT par canal
- Barres horizontales : CSAT par catégorie (du meilleur au pire)

### Message business
> Relier performance opérationnelle et satisfaction client.

---

## Page 6 — Modèle ML

### Visuels à construire
- Jauge : AUC-ROC du modèle (Min 0, Max 1, Cible 0.75)
- Jauge : Recall classe à risque (Cible > 0.75)
- Barres horizontales : feature importance (15 variables)
- Courbes ROC : 3 modèles comparés
- Histogramme : distribution des scores de risque
- Courbe : AUC par fold (validation temporelle)

### Message business
> Valider la fiabilité du modèle et montrer ce qui prédit le risque.

---

# 3. Mesures DAX à créer

Voici les mesures DAX à implémenter dans Power BI.  
Crée une table `_Mesures` pour les organiser proprement.

In [ ]:
Total Tickets = COUNTROWS(fact_support_analytics)

Taux SLA Breach =
DIVIDE(
    CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[sla_breach] = 1),
    [Total Tickets]
)

Taux Backlog =
DIVIDE(
    CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[in_backlog] = 1),
    [Total Tickets]
)

Taux Escalade =
DIVIDE(
    CALCULATE(COUNTROWS(fact_support_analytics), fact_support_analytics[statut] = "Escaladé"),
    [Total Tickets]
)

CSAT Moyen =
AVERAGEX(
    FILTER(fact_support_analytics, NOT(ISBLANK(fact_support_analytics[csat]))),
    fact_support_analytics[csat]
)

First Response Moy = AVERAGE(fact_support_analytics[first_response_heures])

Resolution Moy = AVERAGE(fact_support_analytics[resolution_heures])

Tickets Risque Rouge =
CALCULATE(
    COUNTROWS(tickets_risque_scores),
    SEARCH("ROUGE", tickets_risque_scores[niveau_alerte], 1, 0) > 0
)

Score Risque Moy = AVERAGE(tickets_risque_scores[score_risque])

SLA Breach MoM =
VAR curr = [Taux SLA Breach]
VAR prev = CALCULATE([Taux SLA Breach], PREVIOUSMONTH(Calendrier[Date]))
RETURN DIVIDE(curr - prev, prev)

Tickets Ouverts =
CALCULATE(
    COUNTROWS(fact_support_analytics),
    fact_support_analytics[statut] IN {"En cours", "Backlog"}
)

Quadrant Agent =
SWITCH(TRUE(),
    [Taux SLA Breach] > AVERAGEX(ALL(fact_support_analytics[agent_id]), [Taux SLA Breach]) &&
    [CSAT Moyen] < AVERAGEX(ALL(fact_support_analytics[agent_id]), [CSAT Moyen]),
        "Coaching urgent",
    [Taux SLA Breach] <= AVERAGEX(ALL(fact_support_analytics[agent_id]), [Taux SLA Breach]) &&
    [CSAT Moyen] >= AVERAGEX(ALL(fact_support_analytics[agent_id]), [CSAT Moyen]),
        "Top performer",
    "Moyen"
)

---

# 4. Table Calendrier

Crée une table Calendrier dans Power BI pour gérer les filtres temporels.

In [ ]:
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date],"MMMM","fr-FR"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semaine",      WEEKNUM([Date]),
    "Jour_Semaine", FORMAT([Date],"dddd","fr-FR"),
    "Est_Weekend",  IF(WEEKDAY([Date],2) >= 6, 1, 0)
)

---

# 5. Recommandations de design

## Principes visuels
- 4 à 6 visuels maximum par page
- KPI Cards en haut de page
- Tendances et courbes au centre
- Détails et tableaux en bas
- Palette cohérente : violet (#534AB7) · vert (#1D9E75) · orange (#EF9F27) · rouge (#E24B4A)
- Peu de texte dans les visuels

## Filtres recommandés
- Période (Calendrier[Annee_Mois])
- Pays
- Canal
- Catégorie
- Tier de l'agent
- Niveau d'alerte ML

## Erreurs à éviter
- Trop de couleurs différentes
- Tableaux sans formatage conditionnel
- KPI sans cible visible (objectif ou benchmark)
- Graphiques non triés

---

# 6. ✏️ Storytelling exécutif

Rédige une synthèse de 3 à 5 lignes que tu présenterais à M. KOUAME pour résumer les résultats du dashboard.

Structure suggérée :
1. **Performance globale** — taux de breach et CSAT
2. **Points critiques identifiés** — catégories et agents à problèmes
3. **Ce que le modèle ML apporte** — alertes précoces
4. **Actions prioritaires recommandées**

### ✏️ Ta synthèse

*(rédige ici)*

---

# 7. ✏️ Recommandations business

Propose **3 recommandations concrètes et chiffrées** à partir de tes analyses.

### Recommandation 1
*(titre)* : *(description et justification chiffrée)*

### Recommandation 2
*(titre)* : *(description et justification chiffrée)*

### Recommandation 3
*(titre)* : *(description et justification chiffrée)*

---

# 📌 Conclusion

Le Notebook 4 clôture la partie analytique "classique" du projet en faisant passer l'apprenant :
- de la donnée brute,
- au dataset nettoyé,
- puis aux KPIs SQL,
- et enfin au **pilotage visuel par dashboard**.

---

➡️ **Prochaine étape : Notebook 5 — Machine Learning — Détection de tickets à risque**

---

*DataProjectLab — apprendre la data sur des cas concrets, structurés et orientés métier.*